## TASK 1 Schema Design and Data Ingestion ##

In [1]:
import psycopg2
from typing import Optional, Generator, Any, List, Dict
from pathlib import Path


In [2]:
import os
import glob
import json
import logging
import io
import csv
import pandas as pd
import numpy as np
import xml.etree.cElementTree as ET
from datetime import datetime

In [3]:
class PostgresConnector:
    def __init__(
        self,
        dbname: str = "db_berlin",
        user: str = "dia_user",
        password: str = "dia",
        host: str = "localhost",
        port: int = 5434,
    ):
        self.dbname = dbname
        self.user = user
        self.password = password
        self.host = host
        self.port = port
        self.connection: Optional[psycopg2.extensions.connection] = None

    def connect(self):
        if not self.connection:
            self.connection = psycopg2.connect(
                dbname=self.dbname,
                user=self.user,
                password=self.password,
                host=self.host,
                port=self.port
            )
        return self.connection

    def close(self):
        if self.connection:
            self.connection.close()
            self.connection = None

In [4]:
# Data Directory
DATA_DIR = os.path.join(os.getcwd(), "DBahn-berlin")

if not DATA_DIR: raise FileNotFoundError("Could not find 'DBahn-berlin'.")

### TASK 1.1  Star Schema  ### 

In [5]:
# DATABASE SCHEMA CREATION
def init_db(connector: PostgresConnector):
    """
    Creates Dimensions, Fact Table, and Temporary Staging Tables.
    """
    schema_script = """

    DROP TABLE IF EXISTS fact_train_stops CASCADE;
    DROP TABLE IF EXISTS dim_stations CASCADE;
    DROP TABLE IF EXISTS dim_trains CASCADE;
    DROP TABLE IF EXISTS dim_time CASCADE;
    
    -- 1. Dimensions & Fact Table
    CREATE TABLE IF NOT EXISTS dim_stations (
        eva_number BIGINT PRIMARY KEY,
        station_name VARCHAR(255),
        latitude DECIMAL(9, 6),
        longitude DECIMAL(9, 6),
        category INTEGER
    );

    CREATE TABLE IF NOT EXISTS dim_trains (
        train_key BIGSERIAL PRIMARY KEY,
        train_category VARCHAR(10),
        train_number VARCHAR(50),
        line_name VARCHAR(20),
        owner VARCHAR(50),
        CONSTRAINT unique_train_meta UNIQUE (train_category, train_number, line_name)
    );

    CREATE TABLE IF NOT EXISTS dim_time (
        time_id BIGINT PRIMARY KEY,
        full_date DATE,
        year INTEGER,
        month INTEGER,
        day INTEGER,
        hour INTEGER,
        day_name VARCHAR(10),
        is_peak_hour BOOLEAN
    );

    CREATE TABLE IF NOT EXISTS fact_train_stops (
        stop_id VARCHAR(255) PRIMARY KEY,
        station_eva_number BIGINT,
        train_key BIGINT,
        time_id BIGINT,
        planned_arrival TIMESTAMP,
        actual_arrival TIMESTAMP,
        arrival_delay_min DECIMAL(10, 2) DEFAULT 0,
        planned_departure TIMESTAMP,
        actual_departure TIMESTAMP,
        departure_delay_min DECIMAL(10, 2) DEFAULT 0,
        platform VARCHAR(20),
        is_cancelled BOOLEAN DEFAULT FALSE
    );
    
    -- 2. Performance Indexes
    CREATE INDEX IF NOT EXISTS idx_fact_station ON fact_train_stops(station_eva_number);
    CREATE INDEX IF NOT EXISTS idx_fact_delay ON fact_train_stops(arrival_delay_min);

    -- 3. TEMPORARY TABLES (For ELT Staging)
    DROP TABLE IF EXISTS temp_changes;
    CREATE UNLOGGED TABLE temp_changes (
        stop_id VARCHAR(255), 
        actual_arrival TIMESTAMP,
        actual_departure TIMESTAMP,
        changed_platform VARCHAR(20),
        is_cancelled BOOLEAN
    );

    -- Stores a batch of raw timetable data before merging
    DROP TABLE IF EXISTS staging_planned;
    CREATE UNLOGGED TABLE staging_planned (
        stop_id VARCHAR(255),
        station_name VARCHAR(255),
        planned_arrival TIMESTAMP,
        planned_departure TIMESTAMP,
        planned_platform VARCHAR(20),
        train_category VARCHAR(10),
        train_number VARCHAR(50),
        line_name VARCHAR(20),
        owner VARCHAR(50)
    );
    """
    conn = connector.connect()
    with conn.cursor() as cur:
        cur.execute(schema_script)
        conn.commit()
        print("Database schema initialized.")

### TASK 1.2  ### 

In [6]:
def change_generator(file_list: list) -> Generator[dict, None, None]:
    """Yields change data one stop at a time across multiple files."""
    for fpath in file_list:
        try:
            tree = ET.parse(fpath)
            for s in tree.getroot().findall('s'):
                ar, dp = s.find('ar'), s.find('dp')
                
                # FIX: Ensure cancelled is a boolean, not the timestamp string 'clt'
                is_cancelled = False
                if ar is not None and ar.get('clt') is not None:
                    is_cancelled = True
                elif dp is not None and dp.get('clt') is not None:
                    is_cancelled = True

                yield {
                    'stop_id': s.get('id'),
                    'a_arr': parse_timestamp(ar.get('ct')) if ar is not None else None,
                    'a_dep': parse_timestamp(dp.get('ct')) if dp is not None else None,
                    'c_plat': ar.get('cp') if ar is not None else (dp.get('cp') if dp is not None else None),
                    'cancelled': is_cancelled
                }
        except Exception:
            continue

def timetable_generator(file_list: list, changes_map: dict) -> Generator[dict, None, None]:
    """Yields processed timetable records merged with changes data."""
    for fpath in file_list:
        try:
            tree = ET.parse(fpath)
            root = tree.getroot()
            station_name = root.get('station')
            for s in root.findall('s'):
                stop_id = s.get('id')
                tl = s.find('tl')
                if tl is None: continue

                ar, dp = s.find('ar'), s.find('dp')
                p_arr = parse_timestamp(ar.get('pt')) if ar is not None else None
                p_dep = parse_timestamp(dp.get('pt')) if dp is not None else None
                if not p_arr and not p_dep: continue
                
                # find platform and line
                p_plat = (ar.get('pp') if ar is not None else None) or (dp.get('pp') if dp is not None else None)
                line_name = (ar.get('l') if ar is not None else None) or (dp.get('l') if dp is not None else None)
                
                # Merge 
                change = changes_map.get(stop_id, {})
                actual_arr = change.get('a_arr') or p_arr
                actual_dep = change.get('a_dep') or p_dep
                platform = change.get('c_plat') or p_plat
                is_cancelled = change.get('cancelled', False)

                yield {
                    "stop_id": stop_id,
                    "station_name": station_name,
                    "train_meta": (tl.get('c'), tl.get('n'), line_name, tl.get('o')),
                    "time_meta": p_arr or p_dep,
                    "payload": (
                        stop_id,                                              # 0
                        p_arr,                                               # 1
                        actual_arr,                                          # 2
                        (actual_arr - p_arr).total_seconds() / 60 if p_arr and actual_arr else 0, # 3
                        p_dep,                                               # 4
                        actual_dep,                                          # 5
                        (actual_dep - p_dep).total_seconds() / 60 if p_dep and actual_dep else 0, # 6
                        platform,                                            # 7
                        is_cancelled                                         # 8
                    )
                }
        except Exception as e:
            print(f"Error parsing file {fpath}: {e}")
            continue

def parse_timestamp(ts_str):
    if not ts_str: return None
    try:
        return datetime.strptime(ts_str, '%y%m%d%H%M')
    except ValueError:
        return None

In [7]:
%%time
def main():
    db = PostgresConnector()
    
    # create db tables
    init_db(db)

    # load stations
    
    current_dir = Path(os.getcwd())
    project_root = current_dir.parent
    
    json_path = project_root / 'data' / 'DBahn-berlin' / 'station_data.json'
    DATA_DIR = project_root / 'data' / 'DBahn-berlin'
    print(DATA_DIR)
    print(json_path)
    with open(json_path, 'r', encoding='utf-8') as f:
        station_data = json.load(f)
    
    stations_to_insert = []
    for entry in station_data.get('result', []):
        name = entry.get('name')
        cat = entry.get('category')
        for eva in entry.get('evaNumbers', []):
            if eva.get('isMain'):
                coords = eva.get('geographicCoordinates', {}).get('coordinates', [None, None])
                stations_to_insert.append((int(eva['number']), name, coords[1], coords[0], cat))

    conn = db.connect()
    with conn.cursor() as cur:
        cur.executemany("""
            INSERT INTO dim_stations (eva_number, station_name, latitude, longitude, category)
            VALUES (%s, %s, %s, %s, %s) ON CONFLICT DO NOTHING
        """, stations_to_insert)
        conn.commit()
        
        cur.execute("SELECT station_name, eva_number FROM dim_stations")
        station_lookup = dict(cur.fetchall())


    # recursive=True and ** ->> find files in all subfolders
    change_files = glob.glob(os.path.join(DATA_DIR, 'timetable_changes', '**', '*.xml'), recursive=True)
    print(f"Found {len(change_files)} change files.") 
    changes_map = {item['stop_id']: item for item in change_generator(change_files)}
    
    timetable_files = glob.glob(os.path.join(DATA_DIR, 'timetables', '**', '*.xml'), recursive=True)
    print(f"Found {len(timetable_files)} timetable files.") 
    records_gen = timetable_generator(timetable_files, changes_map)

    with conn.cursor() as cur:
        count = 0
        for record in records_gen:
            # Dim Trains 
            cur.execute("""
                INSERT INTO dim_trains (train_category, train_number, line_name, owner)
                VALUES (%s, %s, %s, %s) ON CONFLICT (train_category, train_number, line_name) DO NOTHING RETURNING train_key
            """, record['train_meta'])
            res = cur.fetchone()
            if res:
                train_key = res[0]
            else:
                cur.execute("""
                    SELECT train_key FROM dim_trains 
                    WHERE train_category=%s AND train_number=%s AND line_name IS NOT DISTINCT FROM %s
                """, record['train_meta'][:3])
                train_key = cur.fetchone()[0]

            # Dim Time
            t = record['time_meta']
            time_id = int(t.strftime('%Y%m%d%H'))
            cur.execute("""
                INSERT INTO dim_time (time_id, full_date, year, month, day, hour, day_name, is_peak_hour)
                VALUES (%s, %s, %s, %s, %s, %s, %s, %s) ON CONFLICT DO NOTHING
            """, (time_id, t.date(), t.year, t.month, t.day, t.hour, t.strftime('%A'), (7 <= t.hour < 9) or (17 <= t.hour < 19)))

            # Fact Table
            eva = station_lookup.get(record['station_name'])
            
            # cols
            fact_values = (
                record['payload'][0],  # stop_id (index 0 of payload)
                eva,                   # station_eva_number
                train_key,             # train_key
                time_id,               # time_id
                record['payload'][1],  # planned_arrival (p_arr)
                record['payload'][2],  # actual_arrival (actual_arr)
                record['payload'][3],  # arrival_delay_min
                record['payload'][4],  # planned_departure (p_dep)
                record['payload'][5],  # actual_departure (actual_dep)
                record['payload'][6],  # departure_delay_min
                record['payload'][7],  # platform
                record['payload'][8]   # is_cancelled
            )

            cur.execute("""
                INSERT INTO fact_train_stops (
                    stop_id, station_eva_number, train_key, time_id, 
                    planned_arrival, actual_arrival, arrival_delay_min,
                    planned_departure, actual_departure, departure_delay_min,
                    platform, is_cancelled
                ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
                ON CONFLICT (stop_id) DO NOTHING
            """, fact_values)
            # Commit every 1000 records for performance
            count += 1
            if count % 1000 == 0:
                conn.commit()
                
        conn.commit()

    db.close()
    print(f"ETL Pipeline Finished. Processed {count} stops.")

if __name__ == "__main__":
    main()

Database schema initialized.
/Users/aycaaksoy/Dev/path/to/your/data/DBahn-berlin
/Users/aycaaksoy/Dev/path/to/your/data/DBahn-berlin/station_data.json
Found 540668 change files.
Found 135692 timetable files.
ETL Pipeline Finished. Processed 2104080 stops.
CPU times: user 9min 2s, sys: 8min 58s, total: 18min 1s
Wall time: 52min 45s
